# Gerando CSV 

In [6]:
import pandas as pd
from collections import defaultdict
import plotly.graph_objects as go
import sys
import matplotlib.pyplot as plt
import glob
import os
import numpy as np
import re
print(sys.executable)


c:\Users\carlo\Documents\Algoritmos\yolooo\.venv\Scripts\python.exe


In [7]:
def process_myfxbook_csv(file_path):
    """
    Processa arquivos CSV do Myfxbook com diferentes formatos
    e retorna um DataFrame padronizado
    """
    print(f"Processando arquivo: {file_path}")
    
    # Tentar ler o arquivo com diferentes codificações
    encodings = ['utf-8', 'latin1', 'windows-1252', 'iso-8859-1']
    lines = None
    
    for encoding in encodings:
        try:
            with open(file_path, 'r', encoding=encoding) as f:
                lines = f.readlines()
            break
        except UnicodeDecodeError:
            continue
    
    if lines is None:
        print(f"Erro: Não foi possível ler o arquivo {file_path} com nenhuma codificação conhecida")
        return None
    
    data = []
    skip_next = False
    
    for i, line in enumerate(lines):
        # Pular cabeçalhos e linhas vazias
        if i == 0 or not line.strip() or 'Date' in line and 'Open' in line:
            continue
            
        # Corrigir problemas comuns
        line = line.strip()
        
        # Remover vírgula final se existir
        if line.endswith(','):
            line = line[:-1]
            
        # Dividir a linha
        parts = line.split(',')
        
        # Caso especial: quando a data vem entre aspas
        if len(parts) > 5 and '"' in line:
            quoted_parts = line.split('"')
            if len(quoted_parts) >= 3:
                timestamp = quoted_parts[1].strip()
                values = quoted_parts[2].strip(',').split(',')[:4]
                parts = [timestamp] + values
        
        # Verificar se temos dados suficientes
        if len(parts) >= 5:
            timestamp = parts[0].strip()
            
            # Tratar casos onde timestamp não tem hora
            if len(timestamp.split()) == 1:
                timestamp += " 00:00"  # Adicionar meia-noite como padrão
            
            # Extrair valores numéricos
            try:
                # Tentar converter os próximos 4 valores
                open_val = float(parts[1].strip())
                high_val = float(parts[2].strip())
                low_val = float(parts[3].strip())
                close_val = float(parts[4].strip())
                
                data.append([timestamp, open_val, high_val, low_val, close_val])
            except (ValueError, IndexError) as e:
                print(f"Ignorando linha {i+1}: {line} | Erro: {str(e)}")
                continue
    
    # Criar DataFrame
    if not data:
        print("Nenhum dado válido encontrado no arquivo")
        return None
        
    df = pd.DataFrame(data, columns=['timestamp', 'open', 'high', 'low', 'close'])
    
    # Converter timestamp para datetime
    try:
        df['timestamp'] = pd.to_datetime(df['timestamp'], format='%m/%d/%Y %H:%M')
    except:
        try:
            df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y-%m-%d %H:%M:%S')
        except:
            print("Formato de data desconhecido. Tentando parser genérico...")
            df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    
    # Remover linhas com datas inválidas
    df = df.dropna(subset=['timestamp'])
    
    print(f"Processado: {len(df)} registros válidos")
    return df

In [8]:
def merge_and_clean_csv_files():
    """
    Processa todos os arquivos CSV relevantes, faz merge e remove o arquivo adicional
    """
    # Encontrar todos os arquivos CSV relevantes
    csv_files = glob.glob('EURUSD_historical_data*.csv')
    
    if not csv_files:
        print("Nenhum arquivo CSV encontrado")
        return None
    
    print(f"Arquivos encontrados: {csv_files}")
    
    # Processar cada arquivo
    dfs = []
    for file in csv_files:
        df = process_myfxbook_csv(file)
        if df is not None and not df.empty:
            dfs.append(df)
    
    if not dfs:
        print("Nenhum dado válido processado")
        return None
    
    # Combinar todos os DataFrames
    combined_df = pd.concat(dfs, ignore_index=True)
    
    # Remover duplicatas mantendo o último registro
    combined_df = combined_df.sort_values('timestamp')
    combined_df = combined_df.drop_duplicates(subset=['timestamp'], keep='last')
    
    print(f"\nDataset combinado: {len(combined_df)} registros únicos")
    print(f"Período: {combined_df['timestamp'].min()} a {combined_df['timestamp'].max()}")
    
    # Salvar o arquivo combinado principal
    main_file = 'EURUSD_historical_data.csv'
    combined_df.to_csv(main_file, index=False)
    print(f"\nDataset final salvo em: {main_file}")
    
    # Remover arquivos adicionais
    for file in csv_files:
        if file != main_file:
            os.remove(file)
            print(f"Arquivo removido: {file}")
    
    return combined_df

In [9]:
process_myfxbook_csv('EURUSD_historical_data.csv')
process_myfxbook_csv('EURUSD_historical_data (1).csv')
df = merge_and_clean_csv_files()
if df is not None:
    # Exibir as primeiras linhas do DataFrame final
    print(df.head())

Processando arquivo: EURUSD_historical_data.csv
Processado: 720 registros válidos
Processando arquivo: EURUSD_historical_data (1).csv
Processado: 720 registros válidos
Arquivos encontrados: ['EURUSD_historical_data (1).csv', 'EURUSD_historical_data.csv']
Processando arquivo: EURUSD_historical_data (1).csv
Processado: 720 registros válidos
Processando arquivo: EURUSD_historical_data.csv
Processado: 720 registros válidos

Dataset combinado: 720 registros únicos
Período: 2024-12-13 12:00:00 a 2025-05-30 00:00:00

Dataset final salvo em: EURUSD_historical_data.csv
Arquivo removido: EURUSD_historical_data (1).csv
              timestamp     open     high      low    close
720 2024-12-13 12:00:00  1.04889  1.05241  1.04822  1.04889
1   2024-12-13 16:00:00  1.04889  1.05061  1.04867  1.04947
722 2024-12-13 20:00:00  1.04953  1.04977  1.04927  1.04969
723 2024-12-15 20:00:00  1.04856  1.05110  1.04831  1.05095
724 2024-12-16 00:00:00  1.05097  1.05229  1.05093  1.05115


In [10]:
print("Processamento concluído com sucesso!")
print(df.head())

Processamento concluído com sucesso!
              timestamp     open     high      low    close
720 2024-12-13 12:00:00  1.04889  1.05241  1.04822  1.04889
1   2024-12-13 16:00:00  1.04889  1.05061  1.04867  1.04947
722 2024-12-13 20:00:00  1.04953  1.04977  1.04927  1.04969
723 2024-12-15 20:00:00  1.04856  1.05110  1.04831  1.05095
724 2024-12-16 00:00:00  1.05097  1.05229  1.05093  1.05115


In [11]:
'''with open("EURUSD_historical_data.csv", 'r') as f:
    lines = f.readlines()'''

'with open("EURUSD_historical_data.csv", \'r\') as f:\n    lines = f.readlines()'

In [12]:
'''data = []
for line in lines[1:]:  # Pular cabeçalhos
    # Remover vírgula final se existir
    if line.strip().endswith(','):
        line = line.strip()[:-1]
    
    parts = line.strip().split(',')
    
    # Verificar se temos pelo menos 5 colunas
    if len(parts) >= 5:
        # Juntar data e hora
        timestamp = parts[0].strip()
        
        # Verificar se o timestamp inclui o valor de abertura
        if ' ' not in timestamp:
            # Se não tiver espaço, tentar dividir de outra forma
            timestamp = parts[0] + ' ' + parts[1]
            values = parts[2:6]
        else:
            values = parts[1:5]
        
        # Converter valores numéricos
        try:
            open_val = float(values[0])
            high_val = float(values[1])
            low_val = float(values[2])
            close_val = float(values[3])
            data.append([timestamp, open_val, high_val, low_val, close_val])
        except (ValueError, IndexError):
            continue'''

"data = []\nfor line in lines[1:]:  # Pular cabeçalhos\n    # Remover vírgula final se existir\n    if line.strip().endswith(','):\n        line = line.strip()[:-1]\n\n    parts = line.strip().split(',')\n\n    # Verificar se temos pelo menos 5 colunas\n    if len(parts) >= 5:\n        # Juntar data e hora\n        timestamp = parts[0].strip()\n\n        # Verificar se o timestamp inclui o valor de abertura\n        if ' ' not in timestamp:\n            # Se não tiver espaço, tentar dividir de outra forma\n            timestamp = parts[0] + ' ' + parts[1]\n            values = parts[2:6]\n        else:\n            values = parts[1:5]\n\n        # Converter valores numéricos\n        try:\n            open_val = float(values[0])\n            high_val = float(values[1])\n            low_val = float(values[2])\n            close_val = float(values[3])\n            data.append([timestamp, open_val, high_val, low_val, close_val])\n        except (ValueError, IndexError):\n            conti

In [13]:
'''df = pd.DataFrame(data, columns=['timestamp', 'open', 'high', 'low', 'close'])
'''

"df = pd.DataFrame(data, columns=['timestamp', 'open', 'high', 'low', 'close'])\n"

In [14]:
'''df['timestamp'] = pd.to_datetime(df['timestamp'], format='%m/%d/%Y %H:%M')'''

"df['timestamp'] = pd.to_datetime(df['timestamp'], format='%m/%d/%Y %H:%M')"

In [15]:
print(df.head())

              timestamp     open     high      low    close
720 2024-12-13 12:00:00  1.04889  1.05241  1.04822  1.04889
1   2024-12-13 16:00:00  1.04889  1.05061  1.04867  1.04947
722 2024-12-13 20:00:00  1.04953  1.04977  1.04927  1.04969
723 2024-12-15 20:00:00  1.04856  1.05110  1.04831  1.05095
724 2024-12-16 00:00:00  1.05097  1.05229  1.05093  1.05115


In [16]:
df.columns = [col.strip().lower().replace("(", "").replace(")", "").replace("%", "pct").replace(" ", "_") for col in df.columns]
df["ema_9"] = df["close"].ewm(span=9, adjust=False).mean()
df["ema_21"] = df["close"].ewm(span=21, adjust=False).mean()

In [17]:
print(df.head())


              timestamp     open     high      low    close     ema_9  \
720 2024-12-13 12:00:00  1.04889  1.05241  1.04822  1.04889  1.048890   
1   2024-12-13 16:00:00  1.04889  1.05061  1.04867  1.04947  1.049006   
722 2024-12-13 20:00:00  1.04953  1.04977  1.04927  1.04969  1.049143   
723 2024-12-15 20:00:00  1.04856  1.05110  1.04831  1.05095  1.049504   
724 2024-12-16 00:00:00  1.05097  1.05229  1.05093  1.05115  1.049833   

       ema_21  
720  1.048890  
1    1.048943  
722  1.049011  
723  1.049187  
724  1.049365  


# Aplicando Gráficos

In [18]:
fig = go.Figure()


In [19]:
fig.add_trace(go.Candlestick(
    x=df['timestamp'],
    open=df['open'],
    high=df['high'],
    low=df['low'],
    close=df['close'],
    name='Candlestick'
))

fig.add_trace(go.Scatter(
    x=df['timestamp'],
    y=df['ema_9'],
    line=dict(color='blue', width=1),
    name='EMA 9'
))

fig.add_trace(go.Scatter(
    x=df['timestamp'],
    y=df['ema_21'],
    line=dict(color='red', width=1),
    name='EMA 21'
))

fig.update_layout(
    title='EUR/USD - Candlestick com EMAs',
    xaxis_title='Data',
    yaxis_title='Preço',
    xaxis_rangeslider_visible=False
)

fig.show()


In [20]:
import os
os.makedirs("charts", exist_ok=True)


In [21]:
classes = {
    "engolfo_alta": 0,
    "engolfo_baixa": 1,
    "martelo": 2,
    "pullback": 3,
    "golden cross": 4,
    "death cross": 5
}

window_size = 50
output_dir = "charts"
os.makedirs(output_dir, exist_ok=True)


In [22]:
"""def detectar_padroes(window_df):
    padroes = []

    candles = window_df.iloc[-5:]  # analisamos os 5 últimos candles
    c1, c2, c3, c4, c5 = candles.itertuples()

    # 🟢 Engolfo de Alta
    if c4.close < c4.open and c5.close > c5.open and c5.close > c4.open and c5.open < c4.close:
        padroes.append("engolfo_alta")

    # 🔴 Engolfo de Baixa
    if c4.close > c4.open and c5.close < c5.open and c5.open > c4.close and c5.close < c4.open:
        padroes.append("engolfo_baixa")

    # 🔨 Martelo (na base de uma tendência de baixa)
    corpo = abs(c5.close - c5.open)
    sombra_inferior = min(c5.open, c5.close) - c5.low
    sombra_superior = c5.high - max(c5.open, c5.close)

    if corpo < sombra_inferior and sombra_superior < sombra_inferior / 2:
        padroes.append("martelo")

    # 🔁 Pullback simples (ex: candle volta próximo a EMA 21)
    if abs(c5.low - c5.ema_21) < (0.5/100):  # ajusta conforme o ativo
        padroes.append("pullback")

    ema_9 = window_df["ema_9"].values
    ema_21 = window_df["ema_21"].values

    # Verifica se na última vela houve cruzamento de alta
    if ema_9[-2] < ema_21[-2] and ema_9[-1] > ema_21[-1]:
        padroes.append("golden_cross")
    
    # Exemplo adicional (cruzamento de baixa)
    if ema_9[-2] > ema_21[-2] and ema_9[-1] < ema_21[-1]:
        padroes.append("death_cross")
    
    return padroes
"""


'def detectar_padroes(window_df):\n    padroes = []\n\n    candles = window_df.iloc[-5:]  # analisamos os 5 últimos candles\n    c1, c2, c3, c4, c5 = candles.itertuples()\n\n    # 🟢 Engolfo de Alta\n    if c4.close < c4.open and c5.close > c5.open and c5.close > c4.open and c5.open < c4.close:\n        padroes.append("engolfo_alta")\n\n    # 🔴 Engolfo de Baixa\n    if c4.close > c4.open and c5.close < c5.open and c5.open > c4.close and c5.close < c4.open:\n        padroes.append("engolfo_baixa")\n\n    # 🔨 Martelo (na base de uma tendência de baixa)\n    corpo = abs(c5.close - c5.open)\n    sombra_inferior = min(c5.open, c5.close) - c5.low\n    sombra_superior = c5.high - max(c5.open, c5.close)\n\n    if corpo < sombra_inferior and sombra_superior < sombra_inferior / 2:\n        padroes.append("martelo")\n\n    # 🔁 Pullback simples (ex: candle volta próximo a EMA 21)\n    if abs(c5.low - c5.ema_21) < (0.5/100):  # ajusta conforme o ativo\n        padroes.append("pullback")\n\n    ema_9

In [23]:
def detectar_padroes(window_df):
    padroes = []

    # Calcular EMAs se necessário
    if 'ema_9' not in window_df.columns:
        window_df['ema_9'] = window_df['close'].ewm(span=9, adjust=False).mean()
    if 'ema_21' not in window_df.columns:
        window_df['ema_21'] = window_df['close'].ewm(span=21, adjust=False).mean()
    
    candles = window_df.iloc[-5:]  # últimos 5 candles
    *_, candle_2, candle_1 = candles.itertuples()  # candle_1 = mais recente
    
    # 🟢 Engolfo de Alta
    if (candle_2.close < candle_2.open and           # candle anterior é baixa
        candle_1.close > candle_1.open and           # candle atual é alta
        candle_1.close > candle_2.open and           # fecha acima da abertura anterior
        candle_1.open < candle_2.close):             # abre abaixo do fechamento anterior
        padroes.append("engolfo_alta")
    
    # 🔴 Engolfo de Baixa
    if (candle_2.close > candle_2.open and           # candle anterior é alta
        candle_1.close < candle_1.open and           # candle atual é baixa
        candle_1.open > candle_2.close and           # abre acima do fechamento anterior
        candle_1.close < candle_2.open):             # fecha abaixo da abertura anterior
        padroes.append("engolfo_baixa")
    
    # 🔨 Martelo 
    corpo = abs(candle_1.close - candle_1.open)
    maximo = max(candle_1.open, candle_1.close)
    minimo = min(candle_1.open, candle_1.close)
    sombra_superior = candle_1.high - maximo
    sombra_inferior = minimo - candle_1.low
    
    # Martelo de alta (corpo na parte superior)
    if (sombra_inferior > 2 * corpo and 
        sombra_superior < corpo / 2 and
        candle_1.close > candle_1.open):  # close > open
        padroes.append("martelo")
    
    # 🔁 Pullback 
    margem = candle_1.ema_21 * 0.005  # 0.5% do valor
    if abs(candle_1.low - candle_1.ema_21) < margem:
        padroes.append("pullback")
    
    # ✨ Golden Cross
    if (window_df['ema_9'].iloc[-2] < window_df['ema_21'].iloc[-2] and 
        window_df['ema_9'].iloc[-1] > window_df['ema_21'].iloc[-1]):
        padroes.append("golden cross")
    
    # 💀 Death Cross
    if (window_df['ema_9'].iloc[-2] > window_df['ema_21'].iloc[-2] and 
        window_df['ema_9'].iloc[-1] < window_df['ema_21'].iloc[-1]):
        padroes.append("death cross")
    
    return padroes

In [24]:
for i in range(len(df) - window_size + 1):
    window_df = df.iloc[i:i + window_size]

    fig = go.Figure()

    fig.add_trace(go.Candlestick(
        x=window_df['timestamp'],
        open=window_df['open'],
        high=window_df['high'],
        low=window_df['low'],
        close=window_df['close'],
        name='Candlestick'
    ))

    fig.add_trace(go.Scatter(
        x=window_df['timestamp'],
        y=window_df['ema_9'],
        line=dict(color='blue', width=1),
        name='EMA 9'
    ))

    fig.add_trace(go.Scatter(
        x=window_df['timestamp'],
        y=window_df['ema_21'],
        line=dict(color='red', width=1),
        name='EMA 21'
    ))

    fig.add_shape(
        type="line",
        x0=window_df['timestamp'].iloc[0],
        x1=window_df['timestamp'].iloc[-1],
        y0=window_df['close'].iloc[-1],
        y1=window_df['close'].iloc[-1],
        line=dict(color="green", dash="dash"),
        name="Fechamento"
    )

    fig.update_layout(
        xaxis_rangeslider_visible=False,
        title=f'Janela {i + 1}',
        showlegend=False,
        margin=dict(l=0, r=0, t=30, b=0)
    )

    filename = f"{output_dir}/chart_{i+1:04d}.png"
    fig.write_image(filename, width=800, height=400)
    print(f"Imagem salva: {filename}")

    # 🔍 Detectar padrões e salvar rótulos
    padroes = detectar_padroes(window_df)
    
    for padrao in padroes:
        classe_id = classes.get(padrao, -1)
        if classe_id != -1:
            label_filename = f"{output_dir}/chart_{i+1:04d}.txt"
            yolo_line = f"{classe_id} 0.5 0.5 0.5 0.5\n"  # centro x/y e width/height normalizados
            with open(label_filename, 'a') as label_file:
                label_file.write(yolo_line)
            print(f"[{padrao}] Rótulo adicionado → {label_filename}")

Imagem salva: charts/chart_0001.png
[engolfo_alta] Rótulo adicionado → charts/chart_0001.txt
[pullback] Rótulo adicionado → charts/chart_0001.txt
Imagem salva: charts/chart_0002.png
[pullback] Rótulo adicionado → charts/chart_0002.txt
Imagem salva: charts/chart_0003.png
[pullback] Rótulo adicionado → charts/chart_0003.txt
Imagem salva: charts/chart_0004.png
[pullback] Rótulo adicionado → charts/chart_0004.txt
Imagem salva: charts/chart_0005.png
[engolfo_alta] Rótulo adicionado → charts/chart_0005.txt
[pullback] Rótulo adicionado → charts/chart_0005.txt
Imagem salva: charts/chart_0006.png
[pullback] Rótulo adicionado → charts/chart_0006.txt
Imagem salva: charts/chart_0007.png
[pullback] Rótulo adicionado → charts/chart_0007.txt
Imagem salva: charts/chart_0008.png
[pullback] Rótulo adicionado → charts/chart_0008.txt
[golden cross] Rótulo adicionado → charts/chart_0008.txt
Imagem salva: charts/chart_0009.png
[pullback] Rótulo adicionado → charts/chart_0009.txt
Imagem salva: charts/chart_0